In [1]:
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

True

# Self querying

self-query retriever 要求您安装 lark 软件包。

In [ ]:
# !pip install lark==1.1.9

## 索引

In [2]:
# !pip install pandas openpyxl
# pip install pandas==2.0.3
# pip install openpyxl-3.1.4

In [3]:
import pandas as pd

# 设置文件路径
file_path = './data/douban_top250_from_kaggle.xlsx'

# 读取 Excel 文件
df = pd.read_excel(file_path)

In [4]:
df.head(10)

,序号,片名,评分,导演,主演,一句话点评
0,1,肖申克的救赎,9.7,弗兰克·德拉邦特,蒂姆·罗宾斯,希望让人自由。
1,2,霸王别姬,9.6,陈凯歌,张国荣,风华绝代。
2,3,阿甘正传,9.5,罗伯特·泽米吉斯,汤姆·汉克斯,一部美国近现代史。
3,4,这个杀手不太冷,9.4,吕克·贝松,让·雷诺,怪蜀黍和小萝莉不得不说的故事。
4,5,美丽人生,9.5,罗伯托·贝尼尼,罗伯托·贝尼尼,最美的谎言。
5,6,泰坦尼克号,9.4,詹姆斯·卡梅隆,莱昂纳多·迪卡普里奥,失去的才是永恒的。
6,7,千与千寻,9.4,宫崎骏,柊瑠美,最好的宫崎骏，最好的久石让。
7,8,辛德勒的名单,9.5,史蒂文·斯皮尔伯格,连姆·尼森,拯救一个人，就是拯救整个世界。
8,9,盗梦空间,9.3,克里斯托弗·诺兰,莱昂纳多·迪卡普里奥,诺兰给了我们一场无法盗取的梦。
9,10,忠犬八公的故事,9.4,莱塞·霍尔斯道姆,理查·基尔,永远都不能忘记你所爱的人。


In [5]:
len(df)

250

In [6]:
from langchain_core.documents import Document

# 创建 Document 对象列表
docs = []

for index, row in df[:10].iterrows():
    doc = Document(
        page_content=row['一句话点评'],
        metadata={
            "number": row['序号'],
            "title": row['片名'],
            "rating": row['评分'],
            "director": row['导演'],
            "actors": row['主演']
        }
    )
    docs.append(doc)

docs

[Document(page_content='希望让人自由。', metadata={'number': 1, 'title': '肖申克的救赎', 'rating': 9.7, 'director': '弗兰克·德拉邦特', 'actors': '蒂姆·罗宾斯'}),
 Document(page_content='风华绝代。', metadata={'number': 2, 'title': '霸王别姬', 'rating': 9.6, 'director': '陈凯歌', 'actors': '张国荣'}),
 Document(page_content='一部美国近现代史。', metadata={'number': 3, 'title': '阿甘正传', 'rating': 9.5, 'director': '罗伯特·泽米吉斯', 'actors': '汤姆·汉克斯'}),
 Document(page_content='怪蜀黍和小萝莉不得不说的故事。', metadata={'number': 4, 'title': '这个杀手不太冷', 'rating': 9.4, 'director': '吕克·贝松', 'actors': '让·雷诺'}),
 Document(page_content='最美的谎言。', metadata={'number': 5, 'title': '美丽人生', 'rating': 9.5, 'director': '罗伯托·贝尼尼', 'actors': '罗伯托·贝尼尼'}),
 Document(page_content='失去的才是永恒的。 ', metadata={'number': 6, 'title': '泰坦尼克号', 'rating': 9.4, 'director': '詹姆斯·卡梅隆', 'actors': '莱昂纳多·迪卡普里奥'}),
 Document(page_content='最好的宫崎骏，最好的久石让。 ', metadata={'number': 7, 'title': '千与千寻', 'rating': 9.4, 'director': '宫崎骏', 'actors': '柊瑠美'}),
 Document(page_content='拯救一个人，就是拯救整个世界。', metadat

In [7]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

In [8]:
# from langchain_community.vectorstores import Qdrant

# qdrant = Qdrant.from_documents(
#     docs,
#     embeddings,
#     location=":memory:",  # 本地模式，不使用 Qdrant 服务器，只使用内存存储
#     collection_name="my_documents",  # 集合名称为“my_documents”
# )

# qdrant = Qdrant.from_documents(
#     docs,
#     embeddings,
#     path="./data/local_qdrant", # 本地模式，不使用 Qdrant 服务器，将向量存储在磁盘指定位置
#     collection_name="my_documents",  # 集合名称为“my_documents”
# )

In [9]:
from qdrant_client import QdrantClient
from langchain_community.vectorstores import Qdrant

url = "http://localhost:6333"
client = QdrantClient(url)

collection_name = "my_documents"

qdrant = Qdrant(client, collection_name, embeddings)

D:\Software\anaconda3\envs\py310_LLMAppCourse2\lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The class `Qdrant` was deprecated in LangChain 0.0.37 and will be removed in 0.3.0. An updated version of the class exists in the langchain-qdrant package and should be used instead. To use it run `pip install -U langchain-qdrant` and import as `from langchain_qdrant import Qdrant`.
  warn_deprecated(


In [10]:
qdrant.client

In [11]:
qdrant.client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='my_documents'), CollectionDescription(name='test'), CollectionDescription(name='all_news'), CollectionDescription(name='MINDsmall_train_news'), CollectionDescription(name='MyCollection'), CollectionDescription(name='e41ab92c3f938ddb3e82110becbbce3e')])

In [12]:
qdrant.client.collection_exists(collection_name)

True

In [13]:
qdrant.client.get_collection(collection_name=collection_name)

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, vectors_count=None, indexed_vectors_count=0, points_count=10, segments_count=8, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=None), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0), quantization_config=None), p

In [14]:
from qdrant_client.http.models import Distance, VectorParams

# 向量维度
size = 1536

# 若集合已存在，删除该集合
if qdrant.client.collection_exists(collection_name):
    print(f"集合已存在 | collection_name: {collection_name}")
    qdrant.client.delete_collection(collection_name)
    print(f"集合已删除 | collection_name: {collection_name}")
    
# 创建新集合
if qdrant.client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
                size=size, distance=Distance.COSINE)
):
    print(f"创建集合成功 | collection_name: {collection_name}")
else:
    print(f"创建集合失败 | collection_name: {collection_name}")

集合已存在 | collection_name: my_documents
集合已删除 | collection_name: my_documents
创建集合成功 | collection_name: my_documents


In [15]:
texts = [d.page_content for d in docs]
metadatas = [d.metadata for d in docs]

In [16]:
texts

['希望让人自由。',
 '风华绝代。',
 '一部美国近现代史。',
 '怪蜀黍和小萝莉不得不说的故事。',
 '最美的谎言。',
 '失去的才是永恒的。 ',
 '最好的宫崎骏，最好的久石让。 ',
 '拯救一个人，就是拯救整个世界。',
 '诺兰给了我们一场无法盗取的梦。',
 '永远都不能忘记你所爱的人。']

In [17]:
metadatas

[{'number': 1,
  'title': '肖申克的救赎',
  'rating': 9.7,
  'director': '弗兰克·德拉邦特',
  'actors': '蒂姆·罗宾斯'},
 {'number': 2,
  'title': '霸王别姬',
  'rating': 9.6,
  'director': '陈凯歌',
  'actors': '张国荣'},
 {'number': 3,
  'title': '阿甘正传',
  'rating': 9.5,
  'director': '罗伯特·泽米吉斯',
  'actors': '汤姆·汉克斯'},
 {'number': 4,
  'title': '这个杀手不太冷',
  'rating': 9.4,
  'director': '吕克·贝松',
  'actors': '让·雷诺'},
 {'number': 5,
  'title': '美丽人生',
  'rating': 9.5,
  'director': '罗伯托·贝尼尼',
  'actors': '罗伯托·贝尼尼'},
 {'number': 6,
  'title': '泰坦尼克号',
  'rating': 9.4,
  'director': '詹姆斯·卡梅隆',
  'actors': '莱昂纳多·迪卡普里奥'},
 {'number': 7,
  'title': '千与千寻',
  'rating': 9.4,
  'director': '宫崎骏',
  'actors': '柊瑠美'},
 {'number': 8,
  'title': '辛德勒的名单',
  'rating': 9.5,
  'director': '史蒂文·斯皮尔伯格',
  'actors': '连姆·尼森'},
 {'number': 9,
  'title': '盗梦空间',
  'rating': 9.3,
  'director': '克里斯托弗·诺兰',
  'actors': '莱昂纳多·迪卡普里奥'},
 {'number': 10,
  'title': '忠犬八公的故事',
  'rating': 9.4,
  'director': '莱塞·霍尔斯道姆',
  'actors': '理查·基尔'}]

In [18]:
qdrant.add_texts(texts, metadatas, ids=None, batch_size=64)

['afe1c9f4092a4a9798049baf9875431c',
 '2d0a975939f14710b2ce94bc5c8892cb',
 'f480c797c9cd4674ae717dc0efd3b43f',
 'e830f601319a44dfb070fd5d29ff564d',
 'c1bac65b46194e7fb47306226dbd7ca3',
 '4caa2fdb43994660bbdd1f1da1fd8e28',
 'e3a1c013640f4b6a9dc19ea5582d2c85',
 '0ff2f76454ac46c8abdf6e376a819267',
 'b4f5f50b5fbf4a458e60feed25577bc8',
 '6190d73b17e04cebadea6f849df84128']

In [19]:
# 相似性搜索
query = "自由"
qdrant.similarity_search(query)

[Document(page_content='希望让人自由。', metadata={'actors': '蒂姆·罗宾斯', 'director': '弗兰克·德拉邦特', 'number': 1, 'rating': 9.7, 'title': '肖申克的救赎', '_id': 'afe1c9f4-092a-4a97-9804-9baf9875431c', '_collection_name': 'my_documents'}),
 Document(page_content='最好的宫崎骏，最好的久石让。 ', metadata={'actors': '柊瑠美', 'director': '宫崎骏', 'number': 7, 'rating': 9.4, 'title': '千与千寻', '_id': 'e3a1c013-640f-4b6a-9dc1-9ea5582d2c85', '_collection_name': 'my_documents'}),
 Document(page_content='失去的才是永恒的。 ', metadata={'actors': '莱昂纳多·迪卡普里奥', 'director': '詹姆斯·卡梅隆', 'number': 6, 'rating': 9.4, 'title': '泰坦尼克号', '_id': '4caa2fdb-4399-4660-bbdd-1f1da1fd8e28', '_collection_name': 'my_documents'}),
 Document(page_content='风华绝代。', metadata={'actors': '张国荣', 'director': '陈凯歌', 'number': 2, 'rating': 9.6, 'title': '霸王别姬', '_id': '2d0a9759-39f1-4710-b2ce-94bc5c8892cb', '_collection_name': 'my_documents'})]

In [20]:
# 带分数的相似性搜索
query = "自由"
qdrant.similarity_search_with_score(query)

[(Document(page_content='希望让人自由。', metadata={'actors': '蒂姆·罗宾斯', 'director': '弗兰克·德拉邦特', 'number': 1, 'rating': 9.7, 'title': '肖申克的救赎', '_id': 'afe1c9f4-092a-4a97-9804-9baf9875431c', '_collection_name': 'my_documents'}),
  0.8671145),
 (Document(page_content='最好的宫崎骏，最好的久石让。 ', metadata={'actors': '柊瑠美', 'director': '宫崎骏', 'number': 7, 'rating': 9.4, 'title': '千与千寻', '_id': 'e3a1c013-640f-4b6a-9dc1-9ea5582d2c85', '_collection_name': 'my_documents'}),
  0.7866076),
 (Document(page_content='失去的才是永恒的。 ', metadata={'actors': '莱昂纳多·迪卡普里奥', 'director': '詹姆斯·卡梅隆', 'number': 6, 'rating': 9.4, 'title': '泰坦尼克号', '_id': '4caa2fdb-4399-4660-bbdd-1f1da1fd8e28', '_collection_name': 'my_documents'}),
  0.78577834),
 (Document(page_content='风华绝代。', metadata={'actors': '张国荣', 'director': '陈凯歌', 'number': 2, 'rating': 9.6, 'title': '霸王别姬', '_id': '2d0a9759-39f1-4710-b2ce-94bc5c8892cb', '_collection_name': 'my_documents'}),
  0.7813218)]

In [21]:
# 带分数的相似性搜索
query = ""
qdrant.similarity_search_with_score(query)

[(Document(page_content='风华绝代。', metadata={'actors': '张国荣', 'director': '陈凯歌', 'number': 2, 'rating': 9.6, 'title': '霸王别姬', '_id': '2d0a9759-39f1-4710-b2ce-94bc5c8892cb', '_collection_name': 'my_documents'}),
  0.7100227),
 (Document(page_content='拯救一个人，就是拯救整个世界。', metadata={'actors': '连姆·尼森', 'director': '史蒂文·斯皮尔伯格', 'number': 8, 'rating': 9.5, 'title': '辛德勒的名单', '_id': '0ff2f764-54ac-46c8-abdf-6e376a819267', '_collection_name': 'my_documents'}),
  0.69641584),
 (Document(page_content='希望让人自由。', metadata={'actors': '蒂姆·罗宾斯', 'director': '弗兰克·德拉邦特', 'number': 1, 'rating': 9.7, 'title': '肖申克的救赎', '_id': 'afe1c9f4-092a-4a97-9804-9baf9875431c', '_collection_name': 'my_documents'}),
  0.6950078),
 (Document(page_content='最美的谎言。', metadata={'actors': '罗伯托·贝尼尼', 'director': '罗伯托·贝尼尼', 'number': 5, 'rating': 9.5, 'title': '美丽人生', '_id': 'c1bac65b-4619-4e7f-b473-06226dbd7ca3', '_collection_name': 'my_documents'}),
  0.69152766)]

In [22]:
# 元数据筛选
from qdrant_client.http import models

query_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="metadata.director",
            match=models.MatchValue(
                value='陈凯歌'
            )
        )
    ]
)

# 带分数的相似性搜索
query = ""
qdrant.similarity_search_with_score(query, filter=query_filter)

[(Document(page_content='风华绝代。', metadata={'actors': '张国荣', 'director': '陈凯歌', 'number': 2, 'rating': 9.6, 'title': '霸王别姬', '_id': '2d0a9759-39f1-4710-b2ce-94bc5c8892cb', '_collection_name': 'my_documents'}),
  0.7100227)]

In [23]:
query = "风华绝代"
query_vector = embeddings.embed_query(query)
len(query_vector)

1536

In [24]:
# 元数据筛选
from qdrant_client.http import models

query_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="metadata.director",
            match=models.MatchValue(
                value='陈凯歌'
            )
        )
    ]
)

# 带分数的相似性搜索
qdrant.similarity_search_with_score_by_vector(query_vector, filter=query_filter)

[(Document(page_content='风华绝代。', metadata={'actors': '张国荣', 'director': '陈凯歌', 'number': 2, 'rating': 9.6, 'title': '霸王别姬', '_id': '2d0a9759-39f1-4710-b2ce-94bc5c8892cb', '_collection_name': 'my_documents'}),
  0.9753972)]

## 创建检索器

In [25]:
from langchain.chains.query_constructor.base import AttributeInfo

metadata_field_info = [
    AttributeInfo(
        name="title",
        description="电影名字",
        type="string",
    ),
    AttributeInfo(
        name="rating",
        description="电影评分（1-10）",
        type="float",
    ),
    AttributeInfo(
        name="director",
        description="电影导演的名字",
        type="string",
    ),
    AttributeInfo(
        name="actors",
        description="主要演员",
        type="string",
    ),
]

In [26]:
metadata_field_info

[AttributeInfo(name='title', description='电影名字', type='string'),
 AttributeInfo(name='rating', description='电影评分（1-10）', type='float'),
 AttributeInfo(name='director', description='电影导演的名字', type='string'),
 AttributeInfo(name='actors', description='主要演员', type='string')]

In [27]:
document_content_description = "对电影的一句话点评"

In [28]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0)

In [29]:
from langchain.retrievers.self_query.base import SelfQueryRetriever

retriever = SelfQueryRetriever.from_llm(
    llm, 
    qdrant, 
    document_content_description, 
    metadata_field_info, 
    verbose=True
)

## 测试检索器

In [31]:
retriever.invoke("以自由为主题的电影")

[Document(page_content='希望让人自由。', metadata={'actors': '蒂姆·罗宾斯', 'director': '弗兰克·德拉邦特', 'number': 1, 'rating': 9.7, 'title': '肖申克的救赎', '_id': 'afe1c9f4-092a-4a97-9804-9baf9875431c', '_collection_name': 'my_documents'}),
 Document(page_content='风华绝代。', metadata={'actors': '张国荣', 'director': '陈凯歌', 'number': 2, 'rating': 9.6, 'title': '霸王别姬', '_id': '2d0a9759-39f1-4710-b2ce-94bc5c8892cb', '_collection_name': 'my_documents'}),
 Document(page_content='失去的才是永恒的。 ', metadata={'actors': '莱昂纳多·迪卡普里奥', 'director': '詹姆斯·卡梅隆', 'number': 6, 'rating': 9.4, 'title': '泰坦尼克号', '_id': '4caa2fdb-4399-4660-bbdd-1f1da1fd8e28', '_collection_name': 'my_documents'}),
 Document(page_content='一部美国近现代史。', metadata={'actors': '汤姆·汉克斯', 'director': '罗伯特·泽米吉斯', 'number': 3, 'rating': 9.5, 'title': '阿甘正传', '_id': 'f480c797-c9cd-4674-ae71-7dc0efd3b43f', '_collection_name': 'my_documents'})]

In [32]:
retriever.invoke("我想看一部评分高于 9.6 分的电影")

[Document(page_content='希望让人自由。', metadata={'actors': '蒂姆·罗宾斯', 'director': '弗兰克·德拉邦特', 'number': 1, 'rating': 9.7, 'title': '肖申克的救赎', '_id': 'afe1c9f4-092a-4a97-9804-9baf9875431c', '_collection_name': 'my_documents'})]

In [33]:
retriever.invoke("宫崎骏导演过的评分高于9分的电影")

[Document(page_content='最好的宫崎骏，最好的久石让。 ', metadata={'actors': '柊瑠美', 'director': '宫崎骏', 'number': 7, 'rating': 9.4, 'title': '千与千寻', '_id': 'e3a1c013-640f-4b6a-9dc1-9ea5582d2c85', '_collection_name': 'my_documents'})]

In [34]:
retriever.invoke("一部电影，忘记名字了，导演是陈凯歌、主演是张国荣")

[Document(page_content='风华绝代。', metadata={'actors': '张国荣', 'director': '陈凯歌', 'number': 2, 'rating': 9.6, 'title': '霸王别姬', '_id': '2d0a9759-39f1-4710-b2ce-94bc5c8892cb', '_collection_name': 'my_documents'}),
 Document(page_content='最好的宫崎骏，最好的久石让。 ', metadata={'actors': '柊瑠美', 'director': '宫崎骏', 'number': 7, 'rating': 9.4, 'title': '千与千寻', '_id': 'e3a1c013-640f-4b6a-9dc1-9ea5582d2c85', '_collection_name': 'my_documents'}),
 Document(page_content='最美的谎言。', metadata={'actors': '罗伯托·贝尼尼', 'director': '罗伯托·贝尼尼', 'number': 5, 'rating': 9.5, 'title': '美丽人生', '_id': 'c1bac65b-4619-4e7f-b473-06226dbd7ca3', '_collection_name': 'my_documents'}),
 Document(page_content='永远都不能忘记你所爱的人。', metadata={'actors': '理查·基尔', 'director': '莱塞·霍尔斯道姆', 'number': 10, 'rating': 9.4, 'title': '忠犬八公的故事', '_id': '6190d73b-17e0-4ceb-adea-6f849df84128', '_collection_name': 'my_documents'})]

### Query Constructor

In [35]:
from langchain.chains.query_constructor.base import (
    StructuredQueryOutputParser,
    get_query_constructor_prompt,
)

prompt = get_query_constructor_prompt(
    document_content_description,
    metadata_field_info,
)
output_parser = StructuredQueryOutputParser.from_components()
query_constructor = prompt | llm | output_parser

In [36]:
query = "宫崎骏导演过的评分高于9分的电影"

In [37]:
query_constructor.invoke(query)

StructuredQuery(query='宫崎骏', filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='director', value='宫崎骏'), Comparison(comparator=<Comparator.GT: 'gt'>, attribute='rating', value=9)]), limit=None)

### Query Translator

In [38]:
retriever.structured_query_translator

In [39]:
from langchain_community.query_constructors.qdrant import QdrantTranslator

structured_query_translator = QdrantTranslator()

TypeError: QdrantTranslator.__init__() missing 1 required positional argument: 'metadata_key'

In [40]:
qdrant.metadata_payload_key

'metadata'

In [41]:
from langchain_community.query_constructors.qdrant import QdrantTranslator

structured_query_translator = QdrantTranslator(
    metadata_key=qdrant.metadata_payload_key)

In [42]:
from langchain_community.query_constructors.qdrant import QdrantTranslator

retriever = SelfQueryRetriever(
    query_constructor=query_constructor,
    vectorstore=qdrant,
    structured_query_translator=structured_query_translator
)

In [43]:
# 这个示例指定了一个查询和复合过滤器
retriever.invoke(query)

[Document(page_content='最好的宫崎骏，最好的久石让。 ', metadata={'actors': '柊瑠美', 'director': '宫崎骏', 'number': 7, 'rating': 9.4, 'title': '千与千寻', '_id': 'e3a1c013-640f-4b6a-9dc1-9ea5582d2c85', '_collection_name': 'my_documents'})]

In [44]:
QdrantTranslator??

Init signature: QdrantTranslator(metadata_key: 'str')
Source:        
class QdrantTranslator(Visitor):
    """Translate `Qdrant` internal query language elements to valid filters."""

    allowed_operators = (
        Operator.AND,
        Operator.OR,
        Operator.NOT,
    )
    """Subset of allowed logical operators."""

    allowed_comparators = (
        Comparator.EQ,
        Comparator.LT,
        Comparator.LTE,
        Comparator.GT,
        Comparator.GTE,
        Comparator.LIKE,
    )
    """Subset of allowed logical comparators."""

    def __init__(self, metadata_key: str):
        self.metadata_key = metadata_key

    def visit_operation(self, operation: Operation) -> rest.Filter:
        try:
            from qdrant_client.http import models as rest
        except ImportError as e:
            raise ImportError(
                "Cannot import qdrant_client. Please install with `pip install "
                "qdrant-client`."
            ) from e

        args = [arg.

#### 分析底层

In [45]:
structured_query = query_constructor.invoke(query)
structured_query

StructuredQuery(query='宫崎骏', filter=Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='director', value='宫崎骏'), Comparison(comparator=<Comparator.GT: 'gt'>, attribute='rating', value=9)]), limit=None)

In [46]:
structured_query.query

'宫崎骏'

In [47]:
structured_query.filter

Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='director', value='宫崎骏'), Comparison(comparator=<Comparator.GT: 'gt'>, attribute='rating', value=9)])

In [48]:
structured_query_translator

In [59]:
query

'宫崎骏导演过的评分高于9分的电影'

In [49]:
structured_query_translator.visit_structured_query(structured_query)

('宫崎骏',
 {'filter': Filter(should=None, min_should=None, must=[FieldCondition(key='metadata.director', match=MatchValue(value='宫崎骏'), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None), FieldCondition(key='metadata.rating', match=None, range=Range(lt=None, gt=9.0, gte=None, lte=None), geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None)], must_not=None)})

In [50]:
structured_query.filter.accept(structured_query_translator)

Filter(should=None, min_should=None, must=[FieldCondition(key='metadata.director', match=MatchValue(value='宫崎骏'), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None), FieldCondition(key='metadata.rating', match=None, range=Range(lt=None, gt=9.0, gte=None, lte=None), geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None)], must_not=None)

In [51]:
structured_query.filter.accept??

Signature: structured_query.filter.accept(visitor: 'Visitor') -> 'Any'
Source:   
    def accept(self, visitor: Visitor) -> Any:
        """Accept a visitor.

        Args:
            visitor: visitor to accept

        Returns:
            result of visiting
        """
        return getattr(visitor, f"visit_{_to_snake_case(self.__class__.__name__)}")(
            self
        )
File:      d:\software\anaconda3\envs\py310_llmappcourse2\lib\site-packages\langchain_core\structured_query.py
Type:      method

In [52]:
structured_query.filter

Operation(operator=<Operator.AND: 'and'>, arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='director', value='宫崎骏'), Comparison(comparator=<Comparator.GT: 'gt'>, attribute='rating', value=9)])

In [53]:
type(structured_query.filter)

langchain_core.structured_query.Operation

In [54]:
structured_query_translator.visit_operation(structured_query.filter)

Filter(should=None, min_should=None, must=[FieldCondition(key='metadata.director', match=MatchValue(value='宫崎骏'), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None), FieldCondition(key='metadata.rating', match=None, range=Range(lt=None, gt=9.0, gte=None, lte=None), geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None)], must_not=None)

In [55]:
structured_query.filter.arguments

[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='director', value='宫崎骏'),
 Comparison(comparator=<Comparator.GT: 'gt'>, attribute='rating', value=9)]

In [56]:
structured_query.filter.arguments[0]

Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='director', value='宫崎骏')

In [57]:
structured_query_translator.visit_comparison(structured_query.filter.arguments[0])

FieldCondition(key='metadata.director', match=MatchValue(value='宫崎骏'), range=None, geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None)

In [58]:
structured_query_translator.visit_comparison(structured_query.filter.arguments[1])

FieldCondition(key='metadata.rating', match=None, range=Range(lt=None, gt=9.0, gte=None, lte=None), geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None)